# ISS Group 24 Modeling

**Before running:**
1. Open the [shared Drive folder](https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing) and click **"Add shortcut to Drive"** (anywhere in your Drive is fine).
2. Select a GPU runtime: *Runtime → Change runtime type → T4 GPU*.
3. Run cells **in order** top-to-bottom. Each stage cell is idempotent — re-running a completed stage skips it automatically via checkpoint detection.

In [1]:
# Enable autoreload so subsequent edits to modeling/*.py are picked up
# without restarting the kernel. Idempotent — safe to re-run.
%load_ext autoreload
%autoreload 2

import os
import sys
import subprocess
from pathlib import Path


In [2]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH = "/content/iss_group_24_src"
USE_GOOGLE_COLAB = False

In [3]:
if USE_GOOGLE_COLAB:
    from google.colab import drive


    def mount_drive() -> str:
        """Mount Google Drive and return the iss_group_24 project root path."""
        drive.mount("/content/drive")
        for candidate in [
            "/content/drive/Shareddrives/iss_group_24",
            "/content/drive/MyDrive/iss_group_24",
        ]:
            if Path(candidate).exists():
                print(f"Drive mounted. Project root: {candidate}")
                return candidate
        raise RuntimeError(
            "Shared folder 'iss_group_24' not found on this Drive.\n"
            f"Open the link and add a shortcut to your Drive: {SHARED_FOLDER_LINK}"
        )

    DRIVE_ROOT = mount_drive()
else:
    DRIVE_ROOT = "."

In [4]:
def setup_repo() -> None:
    """Clone the repo at depth=1, or reset to origin/main if already cloned."""
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(
            ["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"],
            check=True,
        )
        print(f"Repo updated to origin/main: {REPO_LOCAL_PATH}")
    else:
        subprocess.run(
            ["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH],
            check=True,
        )
        print(f"Repo cloned: {REPO_LOCAL_PATH}")
    sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    """Install packages not pre-installed on Colab (torch/torchvision are pre-installed)."""
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pillow>=10.0", "scipy", "matplotlib>=3.7",
        ],
        check=True,
    )
    print("Dependencies ready")


In [5]:
MANIFEST  = f"{DRIVE_ROOT}/dataset/cleaned/manifest.json"
DATA_ROOT = f"{DRIVE_ROOT}/dataset/cleaned"
MODEL_DIR = f"{DRIVE_ROOT}/model"

# Set CWD to the Drive project root so relative paths written by
# train.py (analysis/, manifest parent, etc.) land on the Drive.
os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: /mnt/Onetera/iss_group_24


In [6]:
if USE_GOOGLE_COLAB:    
    setup_repo()
    install_deps()

---
## Step 0 — Build Dataset Manifest (80/20 train/test)

Copies and indexes images into `dataset/cleaned/`. Manifest holds only `train` and `test` splits; the trainer derives K-fold CV from the train pool.
**Run once.** Idempotent: re-runs only when staged files are missing.


In [7]:
import aggregator
from pathlib import Path

train_dir = Path(DATA_ROOT) / 'train'
test_dir  = Path(DATA_ROOT) / 'test'
staged_ok = (
    Path(MANIFEST).exists()
    and train_dir.is_dir() and any(train_dir.iterdir())
    and test_dir.is_dir()  and any(test_dir.iterdir())
)
if staged_ok:
    print(f'Dataset already staged at {DATA_ROOT} — skipping aggregator.')
else:
    print('Running aggregator…')
    aggregator.main()


Dataset already staged at ./dataset/cleaned — skipping aggregator.


---
## Imports & helpers

`resume_from(name_or_path)` returns the value to pass to a `train_stage*` `resume` kwarg. Recognised filenames (resolved against `MODEL_DIR`):
- `last.pt`, `best.pt`
- `stage1_complete.pt`, `stage2_complete.pt`, `stage3_complete.pt`
- `ckpt_s1_epoch{E:03d}.pt`
- `ckpt_s2_epoch{E:03d}.pt`
- `ckpt_s3_epoch{E:03d}_fold{F}.pt`

Absolute paths pass through.


In [8]:
from modeling.train import train_stage1, train_stage2, train_stage3
from modeling.evaluate import run as evaluate_run
from modeling.plot import plot_all_from_jsons

ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

def resume_from(name_or_path: str | None):
    """None → True (auto from last.pt). filename → MODEL_DIR/filename. abs path → as-is."""
    if name_or_path is None:
        return True
    if Path(name_or_path).is_absolute():
        return str(name_or_path)
    return f'{MODEL_DIR}/{name_or_path}'


---
## Shared kwargs

Every hyperparameter is here. Each per-stage cell pulls from this dict; override with cell-local kwargs.


In [9]:
SHARED_KWARGS = dict(
    manifest         = MANIFEST,
    data_root        = DATA_ROOT,
    out_dir          = MODEL_DIR,
    analysis_dir     = ANALYSIS_DIR,

    # CV (Stage 3 only)
    folds            = 3,
    fold_seed        = 42,

    # Episodes / batches
    episodes_per_epoch_s1 = 1500,
    episodes_per_epoch_s2 = 2000,
    episodes_per_epoch_s3 = 3000,
    val_episodes_s1  = 400,
    val_episodes_s2  = 400,
    val_episodes_s3  = 240,
    batch_size       = 16,
    num_workers      = 2,
    n_support        = 4,

    # LR per stage
    lr_heads_s1            = 3e-4,
    lr_heads_s2            = 2e-4,
    lr_backbone_upper_s2   = 7e-5,
    lr_heads_s3            = 1e-4,
    lr_backbone_upper_s3   = 5e-6,
    lr_backbone_lower_s3   = 5e-6,
    weight_decay           = 1e-4,
    grad_clip              = 1.0,
    warmup_frac            = 0.05,

    # Loss / regularisation
    presence_weight     = 1.0,
    attn_loss_weight    = 0.5,
    aux_weight          = 0.1,    # was 0.5; was dominating loss while decoder layer-0 is undertrained
    entropy_reg_weight  = 0.01,
    reg_l2_prior_weight = 0.0,    # was 1e-3; was shrinking DFL expectations toward bin 0 (tiny boxes)
    proto_norm_weight   = 0.0,    # was 1e-3; VICReg + Barlow already prevent prototype collapse
    contrastive         = True,
    contrastive_weight  = 0.1,
    contrastive_temp    = 0.1,
    vicreg              = True,
    vicreg_weight       = 0.05,
    barlow              = True,
    barlow_weight       = 0.005,
    triplet             = False,
    triplet_weight      = 0.1,

    # Curriculum
    neg_prob_s1         = 0.40,
    neg_prob_s2         = 0.45,
    neg_prob_s3         = 0.50,
    hard_neg_ratio_s1   = 0.0,
    hard_neg_ratio_s2   = 0.30,
    hard_neg_ratio_s3   = 0.60,

    # Source-balanced sampler (sums to batch_size)
    # Per-batch source mix (sums to batch_size). Lower vizwiz_base because the
    # target-domain sources (HOTS scenes / InsDet products / VizWiz novel)
    # are what we actually need the model to handle well.
    # User priority: InsDet ≳ vizwiz_novel ≳ HOTS, vizwiz_base least.
    source_mix          = {'vizwiz_base': 2, 'vizwiz_novel': 4, 'hots': 4, 'insdet': 6},

    # Per-source loss multiplier (applied per sample, batch-mean of these scales
    # the dense+presence+aux losses). Bumps target-domain gradients beyond what
    # the batch composition alone provides.
    source_loss_weights = {'vizwiz_base': 0.3, 'vizwiz_novel': 1.7, 'hots': 1.5, 'insdet': 1.8},
    # Warmup → final annealing (interpolated linearly over the first N epochs of each stage).
    source_loss_weights_warmup = {'vizwiz_base': 0.7, 'vizwiz_novel': 1.0, 'hots': 1.0, 'insdet': 1.0},
    source_weight_anneal_epochs_s1 = 3,
    source_weight_anneal_epochs_s2 = 6,
    source_weight_anneal_epochs_s3 = 5,

    # Augmentation
    augment             = True,
    augment_strength    = 1.0,
    multi_scale_s2      = True,
    multi_scale_s3      = True,
    multi_scale_sizes   = (192, 224, 256),

    # Validation TTA
    val_use_tta         = True,
    val_tta_sizes       = (224, 288),

    # Phase-0 last-resort discrimination + anti-collapse losses.
    contrastive_presence_weight    = 0.5,   # presence-aware InfoNCE (positive episodes pull, absent push away)
    feature_spread_weight          = 0.1,   # hinge on cross-attn output spatial std → discriminative conf maps
    use_hardness_weighted_presence = True,  # auto-rebalance presence loss toward worse-classified class

    # AMP + EMA (~2× T4 throughput + +0.5–1.5 mAP from EMA stabilisation).
    use_amp   = True,
    use_ema   = True,
    ema_decay = 0.999,

    # Backbone unfreeze depth.
    stage2_unfreeze_idx = 6,  # was 7 — two extra MobileNet blocks join Stage 2.
    stage3_split_idx    = 6,

    # Checkpoints
    save_stage_completion = True,
    keep_last_n           = 6,

    seed = 42,
)


In [ ]:
# Knobs only the post-stage `evaluate_run` cells use. Splat with
# `**EVAL_KWARGS  # type: ignore[arg-type]` because Pylance collapses the
# heterogeneous dict values into `str | int | float`; the actual call site
# is type-safe at runtime.
EVAL_KWARGS = dict(
    manifest    = MANIFEST,
    data_root   = DATA_ROOT,
    split       = 'test',
    episodes    = 600,
    batch_size  = 8,
    num_workers = 2,
    neg_prob    = 0.5,
    use_tta     = True,
)

---
## Stage 1 — Head warmup (no CV)

**Goal**: Warm up the matcher heads (SupportTokenizer / SupportFusion / cross-attention / detection / presence) on every source. Backbone stays frozen.

- **Trainable**: heads only.
- **LR**: heads `3e-4`.
- **Validation**: sample of the manifest's `test` split (read-only probe).
- **Resume**: auto from `last.pt` if present (set `resume=False` to start fresh).
- **Per-epoch logging** dumps every loss + every metric (overall + per-source) to stdout.
- **Checkpoints**: `ckpt_s1_epoch{E:03d}.pt` per epoch, `stage1_complete.pt` at end.


In [13]:
# Train Stage 1 — head warmup (heads only; backbone frozen).
# Architecture changes ⇒ this retrains Stage 1 from scratch (~30 min on T4 with AMP).
train_stage1(
    **SHARED_KWARGS,
    stage1_epochs = 3,
    resume = False,   # set True or resume_from('ckpt_s1_epoch001.pt') to continue mid-stage
)



=== Stage 1 (head warmup, 5 epochs) ===
┌─ s1 epoch 1/5  (wall=304.4s)
│  lr      : g0=2.82e-04
│  train   : loss=10.3389  qfl=0.6087  neg_qfl=0.0001  centerness=1.2238  dfl=1.2418  giou=2.6452  presence=0.1080
│  train   : attn=0.2449  aux=2.8842  entropy_reg=0.2753  reg_l2_prior=325.1957  proto_norm=7.8368  nt_xent=1.6554
│  train   : vicreg=0.5949  barlow=0.1494  triplet=0.0000  grad_norm=11.0847  n_steps=93
│  val     : n=400  n_pos=193  n_neg=207  map_50=0.0192  map_5095=0.0035  f1_50=0.0426  precision_50=0.0515  recall_50=0.0363
│  val     : iou_mean=0.0630  iou_median=0.0000  iou_p25=0.0000  iou_p75=0.0102  contain_mean=0.1161
│  val     : contain_at_iou_50=0.0363  contain_at_iou_75=0.0000  presence_acc=0.6025  presence_acc_pos=0.8446
│  val     : presence_acc_neg=0.3768  mean_score_pos=0.5617  mean_score_neg=0.5074  presence_auroc=0.6786
│  val     : presence_pr_auc=0.6395  presence_brier=0.2315  frac_pred_near_corner=0.0225  frac_pred_tiny_box=0.1500
│  val     : argmax_cell_

{'best_metric': {'value': 0.0192, 'stage': 1, 'epoch': 1, 'fold': 0},
 'config': {'manifest': './dataset/cleaned/manifest.json',
  'data_root': './dataset/cleaned',
  'out_dir': './model',
  'analysis_dir': './analysis',
  'stage1_epochs': 5,
  'stage2_epochs': 0,
  'stage3_epochs': 0,
  'folds': 3,
  'fold_seed': 42,
  'episodes_per_epoch_s1': 1500,
  'episodes_per_epoch_s2': 1500,
  'episodes_per_epoch_s3': 2000,
  'val_episodes_s1': 400,
  'val_episodes_s2': 400,
  'val_episodes_s3': 240,
  'batch_size': 16,
  'num_workers': 2,
  'n_support': 4,
  'lr_heads_s1': 0.0003,
  'lr_heads_s2': 0.0002,
  'lr_backbone_upper_s2': 5e-05,
  'lr_heads_s3': 0.0001,
  'lr_backbone_upper_s3': 5e-06,
  'lr_backbone_lower_s3': 5e-06,
  'weight_decay': 0.0001,
  'grad_clip': 1.0,
  'warmup_frac': 0.05,
  'presence_weight': 1.0,
  'attn_loss_weight': 0.5,
  'aux_weight': 0.5,
  'entropy_reg_weight': 0.01,
  'reg_l2_prior_weight': 0.001,
  'proto_norm_weight': 0.001,
  'contrastive': True,
  'contrastiv

### Evaluate Stage 1 on the test split

Full kitchen-sink eval (mAP / IoU / containment / presence / per-source / collapse diagnostics) against `dataset/cleaned/test`, using `stage1_complete.pt`. Report at `analysis/stage1/test_eval/test_report.json`.


In [14]:
evaluate_run(
    **EVAL_KWARGS,  # type: ignore[arg-type]
    checkpoint = f'{MODEL_DIR}/stage1_complete.pt',
    report = f'{ANALYSIS_DIR}/stage1/test_eval/test_report.json',
)

report -> analysis/stage1/test_eval/test_report.json
{
  "n": 600,
  "n_pos": 279,
  "n_neg": 321,
  "iou_mean": 0.0917,
  "iou_median": 0.0,
  "iou_p25": 0.0,
  "iou_p75": 0.0973,
  "contain_mean": 0.1449,
  "contain_at_iou_50": 0.0645,
  "contain_at_iou_75": 0.0179,
  "map_50": 0.0192,
  "map_5095": 0.0046,
  "ap_per_iou": {
    "0.50": 0.0192,
    "0.55": 0.0124,
    "0.60": 0.0084,
    "0.65": 0.0037,
    "0.70": 0.0014,
    "0.75": 0.0009,
    "0.80": 0.0003,
    "0.85": 0.0002,
    "0.90": 0.0,
    "0.95": 0.0
  },
  "f1_50": 0.0759,
  "precision_50": 0.0923,
  "recall_50": 0.0645,
  "presence_acc": 0.6217,
  "presence_acc_pos": 0.8208,
  "presence_acc_neg": 0.4486,
  "mean_score_pos": 0.6303,
  "mean_score_neg": 0.4946,
  "presence_auroc": 0.6816,
  "presence_pr_auc": 0.5932,
  "presence_brier": 0.2303,
  "frac_pred_near_corner": 0.0533,
  "frac_pred_tiny_box": 0.3983,
  "conf_map_mean_pos": 0.0262,
  "conf_map_mean_neg": 0.0255,
  "conf_map_std_pos": 0.0343,
  "conf_map_std_neg

{'overall': {'n': 600,
  'n_pos': 279,
  'n_neg': 321,
  'iou_mean': 0.0917,
  'iou_median': 0.0,
  'iou_p25': 0.0,
  'iou_p75': 0.0973,
  'contain_mean': 0.1449,
  'contain_at_iou_50': 0.0645,
  'contain_at_iou_75': 0.0179,
  'map_50': 0.0192,
  'map_5095': 0.0046,
  'ap_per_iou': {'0.50': 0.0192,
   '0.55': 0.0124,
   '0.60': 0.0084,
   '0.65': 0.0037,
   '0.70': 0.0014,
   '0.75': 0.0009,
   '0.80': 0.0003,
   '0.85': 0.0002,
   '0.90': 0.0,
   '0.95': 0.0},
  'f1_50': 0.0759,
  'precision_50': 0.0923,
  'recall_50': 0.0645,
  'presence_acc': 0.6217,
  'presence_acc_pos': 0.8208,
  'presence_acc_neg': 0.4486,
  'mean_score_pos': 0.6303,
  'mean_score_neg': 0.4946,
  'presence_auroc': 0.6816,
  'presence_pr_auc': 0.5932,
  'presence_brier': 0.2303,
  'frac_pred_near_corner': 0.0533,
  'frac_pred_tiny_box': 0.3983,
  'conf_map_mean_pos': 0.0262,
  'conf_map_mean_neg': 0.0255,
  'conf_map_std_pos': 0.0343,
  'conf_map_std_neg': 0.0294,
  'support_proto_norm_mean': 2.7225,
  'support_pr

---
## Stage 2 — Partial unfreeze (no CV)

**Goal**: Adapt backbone features to the few-shot episode distribution by partially unfreezing the upper MobileNetV3 blocks.

- **Trainable**: heads + `backbone.features[7:]`.
- **LR**: heads `2e-4`, backbone upper `5e-5`.
- **Curriculum**: hard-neg ratio 0.30; `neg_prob` 0.45.
- **Multi-scale training**: input size sampled per batch from {192, 224, 256}.
- **Validation**: sample of the manifest's `test` split.
- **Resume**: defaults to `stage1_complete.pt`.
- **Checkpoints**: `ckpt_s2_epoch{E:03d}.pt` per epoch, `stage2_complete.pt` at end.


In [ ]:
# Train Stage 2 — partial unfreeze (heads + features[6:]).
train_stage2(
    **SHARED_KWARGS,
    stage2_epochs = 12,
    resume        = resume_from('stage1_complete.pt'),
)



=== Stage 2 (partial unfreeze, 10 epochs) ===
┌─ s2 epoch 1/10  (wall=319.0s)
│  lr      : g0=4.97e-05  g1=1.99e-04
│  train   : loss=10.8931  qfl=0.6151  neg_qfl=0.0001  centerness=1.2414  dfl=1.3096  giou=2.8234  presence=0.1153
│  train   : attn=0.2859  aux=3.0466  entropy_reg=0.3356  reg_l2_prior=356.0867  proto_norm=8.9857  nt_xent=1.7777
│  train   : vicreg=0.6287  barlow=0.1619  triplet=0.0000  grad_norm=14.6180  n_steps=93
│  val     : n=400  n_pos=193  n_neg=207  map_50=0.0065  map_5095=0.0013  f1_50=0.0530  precision_50=0.0734  recall_50=0.0415
│  val     : iou_mean=0.0530  iou_median=0.0000  iou_p25=0.0000  iou_p75=0.0000  contain_mean=0.0858
│  val     : contain_at_iou_50=0.0415  contain_at_iou_75=0.0000  presence_acc=0.6150  presence_acc_pos=0.7254
│  val     : presence_acc_neg=0.5121  mean_score_pos=0.5306  mean_score_neg=0.4933  presence_auroc=0.6955
│  val     : presence_pr_auc=0.6915  presence_brier=0.2357  frac_pred_near_corner=0.0000  frac_pred_tiny_box=0.0625
│  va

### Evaluate Stage 2 on the test split

Same kitchen-sink eval, using `stage2_complete.pt`. Report at `analysis/stage2/test_eval/test_report.json`.


In [ ]:
evaluate_run(
    **EVAL_KWARGS,  # type: ignore[arg-type]
    checkpoint = f'{MODEL_DIR}/stage2_complete.pt',
    report     = f'{ANALYSIS_DIR}/stage2/test_eval/test_report.json',
)


---
## Stage 3 — Full unfreeze + K-fold rotating CV

**Goal**: End-to-end fine-tuning with cross-fold validation as the main signal. Inside each Stage 3 epoch, the same model rotates through `K=3` folds: train fold-0 → val fold-0 → train fold-1 → val fold-1 → train fold-2 → val fold-2. Per-(epoch, fold) JSON at `analysis/stage3/epoch_NNN/fold_F.json`; per-epoch cross-fold mean / min / max / std at `aggregate.json`.

- **Trainable**: full backbone + heads.
- **LR**: heads `1e-4`, backbone upper `5e-6`, backbone lower `5e-6`.
- **Curriculum**: hard-neg ratio 0.50; `neg_prob` 0.50.
- **Multi-scale training + 2-scale validation TTA** active.
- **Resume**: defaults to `stage2_complete.pt`. Mid-stage resume from `ckpt_s3_epoch{E}_fold{F}.pt` continues at the next fold.
- **Checkpoints**: `ckpt_s3_epoch{E:03d}_fold{F}.pt` per (epoch, fold), `stage3_complete.pt` at end.


In [ ]:
# Train Stage 3 — full unfreeze + K=3 rotating CV.
train_stage3(
    **SHARED_KWARGS,
    stage3_epochs = 20,
    folds         = 3,
    resume        = resume_from('stage2_complete.pt'),
)


### Evaluate Stage 3 on the test split

Final post-training evaluation on the held-out test split using `stage3_complete.pt`. Report at `analysis/stage3/test_eval/test_report.json`.

Use `f'{MODEL_DIR}/best.pt'` instead if you want the best-by-metric checkpoint across all stages.


In [ ]:
evaluate_run(
    **EVAL_KWARGS,  # type: ignore[arg-type]
    checkpoint = f'{MODEL_DIR}/stage3_complete.pt',
    report     = f'{ANALYSIS_DIR}/stage3/test_eval/test_report.json',
)


---
## Generate the offline report

Reads every per-epoch / per-fold / aggregate JSON under `ANALYSIS_DIR` (plus the per-stage `test_eval/test_report.json`) and writes PNGs to `ANALYSIS_DIR/plots/`.


In [ ]:
plot_all_from_jsons(ANALYSIS_DIR)
